In [2]:
from langgraph.graph import START,END,StateGraph
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from typing import TypedDict
import os

In [3]:
# Model creation
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
)

qwen_model = ChatOllama(
    model = 'qwen2.5-1.5b',
    temperature=0
)

In [5]:
@tool
def ticket_booking(user : str) -> str:
    "Based on the user query check the Availability of the ticket and give the answer"
    ticket = [0,1,2,3,4,5,6,7]
    capacity = 10
    if len(ticket) < capacity:
        ticket.append(1)
        return "Ticket Available : ",len(ticket)-capacity
    else:
        return "HouseFull !"

In [ ]:
class AgentStae(TypedDict):
    user : str
    decision : str
    tool_result : str
    answer : str

def agent(state : AgentStae):
    user = state['user']
    prompt = f'''
        You're and AI Agent,
        Check the user query : {user}

        Decide What to do.
        
        If the Question is related to Movie Ticket Booking,
        response exactly : USE_TOOL

        otherwise respond exactly : DIRECT_ANSWER
        '''
    response = gemini_model.invoke(prompt)
    decision = response.content.strip()
    return {"decision" : decision}

def use_tool(state : AgentStae):
    user = state['user']
    prompt = f''' 
    Example :
    If User Query's context is to Movie Ticket Booking,
    query : {user}

    Example :
    I want to Book a Movie Ticket,
    Return,
    If there is seats available in the theater say in the context of "Tickets Booked Succesfully!".
    else "No Tickets Available".
    '''
    response = gemini_model.invoke(prompt)
    movie = response.content.strip()
    result = ticket_booking.invoke({
        "movie" : movie
    })
    return {"tool_result" : result}

def 